In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
WEIGHTS_DIR = "gravnet_binary_classifier_faser"   # folder under get_weights_path()
RUN         = 10000
GPU         = "cuda:0"
NUM_EVENTS  = None      # None → load all events per chunk
SAVE_CACHE  = False     # set True to save inference results to disk

CLASS_NAMES      = ["background", "primary_EM_e"]
NUM_NODE_CLASSES = 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import seaborn as sns
import torch
from pathlib import Path
from sklearn.metrics import (
    confusion_matrix, roc_curve, auc,
    precision_recall_fscore_support, classification_report,
)
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm

from analysis.gravnet.model import NeutrinoGravNetNodesFaser
from analysis.utils.utils import get_torch_path, get_weights_path, get_figures_path

sns.set_style("ticks")
sns.set_context("paper", font_scale=1.2)
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['DejaVu Serif', 'Times New Roman', 'Times'],
    'mathtext.fontset': 'dejavuserif',
    'axes.linewidth': 0.8,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'figure.dpi': 350,
})

COLORS    = ["#4477AA", "#EE6677"]
MARKER_KW = dict(markersize=4, markerfacecolor='white', markeredgewidth=1.2)
C1 = "#353D4C"   # train
C2 = "#E17883"   # val
C3 = "#5691D9"   # recall

device       = torch.device(GPU if torch.cuda.is_available() else "cpu")
weights_path = get_weights_path() / WEIGHTS_DIR
torch_path   = get_torch_path()
figures_path = get_figures_path() / "gravnet_binary_classifier" / WEIGHTS_DIR
figures_path.mkdir(parents=True, exist_ok=True)

print(f"Device  : {device}")
print(f"Weights : {weights_path}")
print(f"Figures : {figures_path}")

__Training curves__

In [ ]:
metrics_path = weights_path / "training_metrics.npz"

if not metrics_path.exists():
    print(f"training_metrics.npz not found — training still in progress.")
else:
    metrics = np.load(metrics_path)
    n_recorded = len(metrics["train_loss"])

    # Infer epoch offset — handles resumed training where npz only has partial history.
    # latest_checkpoint.pt stores the absolute epoch number, so we use it to anchor the x-axis.
    latest_ckpt = weights_path / "latest_checkpoint.pt"
    if latest_ckpt.exists():
        _c          = torch.load(latest_ckpt, map_location="cpu", weights_only=False)
        epoch_end   = _c["epoch"] + 1           # 1-indexed last recorded epoch
        epoch_start = epoch_end - n_recorded + 1
    else:
        epoch_start = 1
        epoch_end   = n_recorded

    epochs  = np.arange(epoch_start, epoch_end + 1)
    best_ep = epoch_start + int(np.argmin(metrics["val_loss"]))

    # Backward-compatible: old runs have val_recall_prim_EM, new runs have val_auc
    auc_key   = "val_auc" if "val_auc" in metrics else "val_recall_prim_EM"
    auc_label = "Val AUC-ROC" if "val_auc" in metrics else "Val recall (primary_EM_e)"
    auc_title = "AUC-ROC (val)" if "val_auc" in metrics else "Recall on primary_EM_e (val)"
    auc_base  = 0.5

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    ax = axes[0]
    ax.plot(epochs, metrics["train_loss"], marker='o', lw=0.8, color=C1, label="Train", **MARKER_KW)
    ax.plot(epochs, metrics["val_loss"],   marker='s', lw=0.8, color=C2, label="Val",   **MARKER_KW)
    ax.axvline(best_ep, color=C2, ls=':', lw=1.0, alpha=0.7, label=f"Best (ep {best_ep})")
    ax.set_yscale("log")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Cross-entropy loss")
    ax.legend(frameon=False); ax.set_title("Loss")
    ax.spines[["top", "right"]].set_visible(False); ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.plot(epochs, metrics["train_acc"], marker='o', lw=0.8, color=C1, label="Train", **MARKER_KW)
    ax.plot(epochs, metrics["val_acc"],   marker='s', lw=0.8, color=C2, label="Val",   **MARKER_KW)
    ax.axvline(best_ep, color=C2, ls=':', lw=1.0, alpha=0.7)
    ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy")
    ax.legend(frameon=False); ax.set_title("Overall accuracy")
    ax.spines[["top", "right"]].set_visible(False); ax.grid(True, alpha=0.3)

    ax = axes[2]
    ax.plot(epochs, metrics[auc_key], marker='s', lw=0.8, color=C3, label=auc_label, **MARKER_KW)
    ax.axhline(auc_base, color='gray', ls='--', lw=0.8, label=f"{auc_base} baseline")
    ax.axvline(best_ep, color=C2, ls=':', lw=1.0, alpha=0.7, label=f"Best (ep {best_ep})")
    ax.set_xlabel("Epoch"); ax.set_ylabel(auc_label)
    ax.legend(frameon=False); ax.set_title(auc_title)
    ax.spines[["top", "right"]].set_visible(False); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(figures_path / "training_curves.png", dpi=350, bbox_inches='tight')
    plt.show()
    print(f"Epochs plotted : {epoch_start}–{epoch_end}  (recorded: {n_recorded})")
    print(f"Best epoch: {best_ep}  val_loss={metrics['val_loss'][best_ep - epoch_start]:.4f}  "
          f"val_acc={metrics['val_acc'][best_ep - epoch_start]:.4f}  "
          f"{auc_key}={metrics[auc_key][best_ep - epoch_start]:.4f}")
    print("Saved: training_curves.png")

__Load data (held-out validation set)__

In [ ]:
def get_str_from_run(run):
    return ["nue", "num", "nut", "nun"][run % 4]

run_str  = get_str_from_run(RUN)
run_path = torch_path / f"{RUN}/pointnetpp_faser_all_events"

chunk_files = sorted(run_path.glob(f"{run_str}_*.pt"))
chunk_files = [f for f in chunk_files
               if "_particle_prob" not in f.stem and "_binary_prob" not in f.stem]
print(f"Run {RUN} ({run_str}): {len(chunk_files)} base chunks found")

dataset = []
for chunk_file in chunk_files:
    chunk_data = torch.load(chunk_file, weights_only=False)
    if NUM_EVENTS is not None:
        chunk_data = chunk_data[:NUM_EVENTS]
    dataset.extend(chunk_data)
    print(f"  {chunk_file.name}: {len(chunk_data)} events")

print(f"\nTotal events loaded: {len(dataset)}")

# Reproducible val split — must match training (random_state=42, test_size=0.2)
_, val_dataset = train_test_split(dataset, test_size=0.2, random_state=42)
print(f"Val set            : {len(val_dataset)} events")

from torch_geometric.loader import DataLoader
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

__Load model and run inference__

In [ ]:
ckpt = torch.load(weights_path / "best_model.pt", map_location=device, weights_only=False)
cfg  = ckpt.get("model_config", {})

model = NeutrinoGravNetNodesFaser(
    input_dim=1, num_node_classes=NUM_NODE_CLASSES, faser_dim=5,
    n_gravstack=cfg.get("n_gravstack", 3),
    out_channels=cfg.get("out_channels", 16),
    n_feature_transform=cfg.get("n_feature_transform", 16),
    k=cfg.get("k", 12),
).to(device)
model.load_state_dict(ckpt["model_state_dict"])

print(f"Checkpoint epoch : {ckpt['epoch'] + 1}")
print(f"Val loss         : {ckpt.get('val_loss', float('nan')):.4f}")
print(f"Val recall pEM   : {ckpt.get('val_recall_prim_EM', float('nan')):.4f}")
print(f"Parameters       : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# Inference — loads from cache if available, saves if SAVE_CACHE=True
_ep        = ckpt['epoch']
_vl        = ckpt.get('val_loss', 0.0)
cache_path = figures_path / f"infer_cache_ep{_ep}_vl{_vl:.4f}.npz"

if cache_path.exists():
    print(f"\nLoading inference from cache: {cache_path.name}")
    _c     = np.load(cache_path)
    y_true = _c["y_true"]; y_pred = _c["y_pred"]; y_prob = _c["y_prob"]
    print(f"Loaded {len(y_true):,} nodes.")
else:
    print("\nRunning inference...")
    model.eval()
    all_targets, all_predictions, all_probabilities = [], [], []
    with torch.no_grad():
        for data in tqdm(val_loader, desc="Inference"):
            data = data.to(device)
            if data.x.size(0) == 0:
                continue
            out  = model(data.x, data.pos, data.batch, data.x_faser)
            prob = torch.softmax(out, dim=1)
            pred = out.argmax(dim=1)
            binary_label = (data.pdg_label == 2).long()
            all_targets.append(binary_label.cpu().numpy())
            all_predictions.append(pred.cpu().numpy())
            all_probabilities.append(prob.cpu().numpy())
    y_true = np.concatenate(all_targets)
    y_pred = np.concatenate(all_predictions)
    y_prob = np.concatenate(all_probabilities)
    if SAVE_CACHE:
        np.savez_compressed(cache_path, y_true=y_true, y_pred=y_pred, y_prob=y_prob)
        print(f"Saved inference cache: {cache_path.name}")

print(f"\nNodes evaluated : {len(y_true):,}")
print(f"Overall accuracy: {(y_true == y_pred).mean():.4f}")

__Per-class metrics__

In [ ]:
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, labels=[0, 1], zero_division=0
)
print(f"{'Class':<20}  {'Precision':>10}  {'Recall':>10}  {'F1':>10}  {'Support':>12}")
print("-" * 68)
for name, p, r, f, s in zip(CLASS_NAMES, precision, recall, f1, support):
    print(f"{name:<20}  {p:>10.4f}  {r:>10.4f}  {f:>10.4f}  {s:>12,}")

__Confusion matrices__

In [ ]:
cm           = confusion_matrix(y_true, y_pred)
cm_norm_row  = cm.astype(float) / cm.sum(axis=1, keepdims=True)
cm_norm_col  = cm.astype(float) / cm.sum(axis=0, keepdims=True)

def make_annot(cm_raw, cm_norm):
    annot = np.empty_like(cm_raw, dtype=object)
    for i in range(cm_raw.shape[0]):
        for j in range(cm_raw.shape[1]):
            annot[i, j] = f"{cm_norm[i, j]:.2f}\n({cm_raw[i, j]:,})"
    return annot

fig, axes = plt.subplots(1, 2, figsize=(9, 4))

sns.heatmap(cm_norm_row, annot=make_annot(cm, cm_norm_row), fmt="", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            vmin=0, vmax=1, cbar_kws={"label": "Recall (row fraction)"}, ax=axes[0])
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")
axes[0].set_title("Row-normalised (recall)"); axes[0].set_aspect("equal")

sns.heatmap(cm_norm_col, annot=make_annot(cm, cm_norm_col), fmt="", cmap="Greens",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            vmin=0, vmax=1, cbar_kws={"label": "Precision (col fraction)"}, ax=axes[1])
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("True")
axes[1].set_title("Column-normalised (precision)"); axes[1].set_aspect("equal")

plt.tight_layout()
plt.savefig(figures_path / "confusion_matrices.png", dpi=350, bbox_inches="tight")
plt.show()

__Softmax probability distributions per true class__

For each true class, histogram of $P(\text{class})$ split by correct vs incorrect predictions.

__Prediction confidence distribution__

Confidence = $P(\text{predicted class})$, always $\geq 0.5$. Split by correct vs incorrect predictions.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
bins = np.linspace(0, 1, 31)

correct_mask = y_pred == y_true
confidence   = y_prob.max(axis=1)  # P(predicted class), always >= 0.5

ax.hist(confidence[correct_mask],  bins=bins, density=True, alpha=0.65,
        color="steelblue", label=f"Correct  ({correct_mask.sum():,})", histtype='stepfilled')
ax.hist(confidence[~correct_mask], bins=bins, density=True, alpha=1.0,
        color="#999999", label=f"Incorrect ({(~correct_mask).sum():,})", histtype='step', linewidth=1.5)

ax.set_xlabel("Confidence $P(\\mathrm{predicted\\ class})$")
ax.set_ylabel("Density")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(figures_path / "confidence_distribution.png", dpi=350, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
bins = np.linspace(0, 1, 31)

for true_idx, (ax, true_name) in enumerate(zip(axes, CLASS_NAMES)):
    mask         = y_true == true_idx
    prob_col     = y_prob[mask, true_idx]
    correct_mask = y_pred[mask] == true_idx
    n_total      = int(mask.sum())   # both histograms share this denominator

    # weights=1/n_total so that correct+incorrect bars at each bin sum to the
    # fraction of true-class nodes in that bin; the two histograms together
    # integrate to 1 over all bins.
    ax.hist(prob_col[correct_mask],  bins=bins, alpha=0.65,
            color=COLORS[true_idx],
            weights=np.ones(int(correct_mask.sum())) / n_total,
            label=f"Correct  ({correct_mask.sum():,})")
    ax.hist(prob_col[~correct_mask], bins=bins, alpha=0.50,
            color="#999999",
            weights=np.ones(int((~correct_mask).sum())) / n_total,
            label=f"Incorrect ({(~correct_mask).sum():,})")
    ax.set_xlabel(f"$P(\\mathtt{{{true_name}}})$", fontsize=10)
    ax.set_ylabel("Fraction of true-class nodes per bin")
    ax.set_title(f"True class: {true_name}  (n\u2009=\u2009{n_total:,})")
    ax.legend(fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)

plt.suptitle(
    "Predicted softmax probability for each true class\n"
    "(normalised by total nodes in each true class; correct + incorrect integrate to 1)",
    y=1.03,
)
plt.tight_layout()
plt.savefig(figures_path / "probability_distributions.png", dpi=350, bbox_inches="tight")
plt.show()


__ROC curve__

In [ ]:
fpr, tpr, _ = roc_curve(y_true, y_prob[:, 1])
roc_auc     = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color=COLORS[1], lw=1.8, label=f"primary_EM_e  (AUC = {roc_auc:.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=0.8, label="Random")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC curve — binary classifier")
ax.legend(fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(figures_path / "roc_curve.png", dpi=350, bbox_inches="tight")
plt.show()
print(f"AUC = {roc_auc:.4f}")

__Optimal threshold (Youden index)__

Finds the threshold maximising $J = \text{TPR} - \text{FPR}$, i.e. the point on the ROC curve closest to the top-left corner.

In [ ]:
fpr, tpr, thresholds = roc_curve(y_true, y_prob[:, 1])

J           = tpr - fpr
optimal_idx = J.argmax()
optimal_threshold = thresholds[optimal_idx]

print(f"Optimal threshold (Youden): {optimal_threshold:.3f}")
print(f"TPR at threshold:           {tpr[optimal_idx]:.3f}")
print(f"FPR at threshold:           {fpr[optimal_idx]:.3f}")
print(f"Default threshold (0.5):    TPR={tpr[np.searchsorted(fpr, (fpr[fpr <= 0.5]).max())]:.3f}  FPR~0.5")

__Event displays — best & worst by node accuracy (2-D)__

Three projections ($xz$, $xy$, $yz$) for the N best and N worst validation events ranked by per-event node accuracy.
Correct predictions: small filled circles, low opacity. Misclassified nodes: diamonds — **fill = predicted class, black edge** — so colour tells you what the node was wrongly called.

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
N_BEST   = 2
N_WORST  = 2
N_RANDOM = 2

C_BG = "#4477AA"   # blue  — background
C_EM = "#44AA77"   # green — primary_EM_e

VIEWS = [
    ("$xz$", 2, 0, "$z$ (mm)", "$x$ (mm)"),
    ("$xy$", 0, 1, "$x$ (mm)", "$y$ (mm)"),
    ("$yz$", 2, 1, "$z$ (mm)", "$y$ (mm)"),
]

# ── Batched inference over val_loader ────────────────────────────────────────
model.eval()
per_event = []

with torch.no_grad():
    for batch_data in tqdm(val_loader, desc="Scoring val events"):
        batch_data = batch_data.to(device)
        out   = model(batch_data.x, batch_data.pos, batch_data.batch, batch_data.x_faser)
        prob  = torch.softmax(out, dim=1).cpu()
        pred  = out.argmax(dim=1).cpu()
        true  = (batch_data.pdg_label == 2).long().cpu()
        pos   = batch_data.pos.cpu()
        batch = batch_data.batch.cpu()
        for g in range(int(batch.max().item()) + 1):
            m = batch == g
            per_event.append({'pred': pred[m].numpy(), 'true': true[m].numpy(),
                               'prob': prob[m].numpy(), 'pos':  pos[m].numpy()})

records = []
for idx, ev in enumerate(per_event):
    data  = val_dataset[idx]
    E_nu  = float(data.E_nu)
    E_roe = float(data.E_roe)
    records.append({'idx': idx, 'acc': float((ev['pred'] == ev['true']).mean()), **ev,
                    'E_nu': E_nu, 'E_roe': E_roe,
                    'inelasticity': E_roe / E_nu if E_nu > 0 else np.nan,
                    'n_nodes': int(data.x.size(0)),
                    'pEM_frac': float((data.pdg_label == 2).float().mean()),
                    'pdg_label': data.pdg_label.numpy()})

acc_arr  = np.array([r['acc']          for r in records])
Enu_arr  = np.array([r['E_nu']         for r in records])
inel_arr = np.array([r['inelasticity'] for r in records])
nn_arr   = np.array([r['n_nodes']      for r in records])
pEM_arr  = np.array([r['pEM_frac']     for r in records])
print(f"Scored {len(records)} events  |  acc {acc_arr.min():.3f}–{acc_arr.max():.3f}")

# ── Select worst / random / best ─────────────────────────────────────────────
records.sort(key=lambda r: r["acc"])
worst = records[:N_WORST]
best  = records[-N_BEST:]
rng   = np.random.default_rng(seed=42)
pool  = [r for r in records if r['idx'] not in {r2['idx'] for r2 in worst + best}]
rand  = list(rng.choice(pool, size=min(N_RANDOM, len(pool)), replace=False))

selected   = worst + rand + best
group_sizes = [N_WORST, N_RANDOM, N_BEST]
group_names = ["Worst", "Random", "Best"]
row_labels  = (
    [f"Worst #{i+1}  acc={r['acc']:.3f}" for i, r in enumerate(worst)] +
    [f"Random #{i+1}  acc={r['acc']:.3f}" for i, r in enumerate(rand)]  +
    [f"Best #{i+1}   acc={r['acc']:.3f}" for i, r in enumerate(best)]
)

# ── Plot ─────────────────────────────────────────────────────────────────────
n_rows = len(selected)
fig, axes = plt.subplots(n_rows, 3, figsize=(13, 3.2 * n_rows),
                         gridspec_kw={"hspace": 0.50, "wspace": 0.35})
if n_rows == 1:
    axes = axes[np.newaxis, :]

sep_after = set()   # row indices after which to draw a separator
running = 0
for g in group_sizes[:-1]:
    running += g
    sep_after.add(running - 1)

for row, (rec, row_lbl) in enumerate(zip(selected, row_labels)):
    pos, pred, true = rec["pos"], rec["pred"], rec["true"]
    correct = pred == true

    for col, (view_title, hcol, vcol, xlabel, ylabel) in enumerate(VIEWS):
        ax = axes[row, col]

        # Correct — circle, colour = predicted class
        for cls_idx, color in enumerate([C_BG, C_EM]):
            mask = (pred == cls_idx) & correct
            if mask.any():
                ax.scatter(pos[mask, hcol], pos[mask, vcol],
                           c=color, s=5, alpha=0.28, linewidths=0,
                           zorder=2, rasterized=True)

        # Misclassified — small X, colour = predicted class
        for cls_idx, color in enumerate([C_BG, C_EM]):
            mask = (pred == cls_idx) & ~correct
            if mask.any():
                ax.scatter(pos[mask, hcol], pos[mask, vcol],
                           c=color, s=10, marker='x', linewidths=0.6,
                           zorder=4, rasterized=True)

        ax.set_xlabel(xlabel, fontsize=8)
        ax.set_ylabel(ylabel, fontsize=8)
        ax.tick_params(labelsize=7)
        ax.spines[["top", "right"]].set_visible(False)
        ax.set_title(f"{row_lbl} · {view_title}" if col == 0 else view_title,
                     fontsize=8, loc="left")

    if row in sep_after:
        for col in range(3):
            axes[row, col].spines["bottom"].set_linestyle((0, (5, 4)))
            axes[row, col].spines["bottom"].set_linewidth(1.0)
            axes[row, col].spines["bottom"].set_color("#888888")
            axes[row, col].spines["bottom"].set_visible(True)

legend_handles = [
    mlines.Line2D([], [], marker='o', color='w', markerfacecolor=C_BG,
                  markersize=6, alpha=0.7, label='predicted background'),
    mlines.Line2D([], [], marker='o', color='w', markerfacecolor=C_EM,
                  markersize=6, alpha=0.7, label='predicted primary_EM_e'),
    mlines.Line2D([], [], marker='x', color='#555555', markersize=6,
                  linestyle='None', markeredgewidth=0.8, label='misclassified'),
]
fig.legend(handles=legend_handles, loc='upper center',
           bbox_to_anchor=(0.5, 1.01), ncol=3, fontsize=8, frameon=False)
plt.savefig(figures_path / "best_worst_events_2d.png", dpi=350, bbox_inches='tight')
plt.show()

__Single event — interactive 3-D display (Plotly)__

Set `EVENT_IDX` to any val-dataset index. Rotate/zoom in the browser.
Colour = predicted class (blue = background, green = primary_EM_e); crosses = misclassified; hover shows $P(\text{primary\_EM\_e})$.

In [ ]:
import plotly.graph_objects as go

# ── Config — change EVENT_IDX to explore different events ───────────────────
EVENT_IDX = 0   # index into val_dataset

# ── Inference on this single event ──────────────────────────────────────────
data   = val_dataset[EVENT_IDX]
data_d = data.clone().to(device)
batch  = (data_d.batch if data_d.batch is not None
          else torch.zeros(data_d.x.size(0), dtype=torch.long, device=device))

model.eval()
with torch.no_grad():
    out  = model(data_d.x, data_d.pos, batch, data_d.x_faser)
    prob = torch.softmax(out, dim=1).cpu().numpy()
    pred = out.argmax(dim=1).cpu().numpy()

true    = (data.pdg_label == 2).long().numpy()
pos     = data.pos.numpy()
correct = pred == true
acc     = correct.mean()

# ── Build traces ─────────────────────────────────────────────────────────────
# (true_cls, pred_cls, label, colour, symbol, size, opacity)
C_BG = "#4477AA"   # blue  — background
C_EM = "#44AA77"   # green — primary_EM_e

trace_configs = [
    (0, 0, "background (correct)",       C_BG, "circle", 2.5, 0.28),
    (1, 1, "primary_EM_e (correct)",     C_EM, "circle", 2.5, 0.28),
    (0, 1, "bg → primary_EM_e",          C_EM, "cross",  6,   0.95),
    (1, 0, "primary_EM_e → background",  C_BG, "cross",  6,   0.95),
]

traces = []
for true_cls, pred_cls, label, color, symbol, size, opacity in trace_configs:
    mask = (true == true_cls) & (pred == pred_cls)
    if not mask.any():
        continue
    idx_arr = np.where(mask)[0]
    hover   = [f"P(pEM) = {prob[i, 1]:.3f}  |  "
               f"true = {'pEM' if true[i] else 'bg'}  "
               f"pred = {'pEM' if pred[i] else 'bg'}"
               for i in idx_arr]
    traces.append(go.Scatter3d(
        x=pos[mask, 2], y=pos[mask, 0], z=pos[mask, 1],
        mode='markers',
        marker=dict(size=size, color=color, opacity=opacity,
                    symbol=symbol, line=dict(width=0)),
        name=f"{label} ({mask.sum():,})",
        text=hover,
        hovertemplate="%{text}<extra></extra>",
    ))

fig3d = go.Figure(data=traces)
fig3d.update_layout(
    title=dict(
        text=f"Val event {EVENT_IDX}  |  acc = {acc:.3f}  |  {len(true):,} nodes  |  "
             f"misclassified = {int((~correct).sum())} ({100*(~correct).mean():.1f}%)",
        font=dict(size=12),
    ),
    scene=dict(
        xaxis=dict(title="z (mm)", backgroundcolor="white", gridcolor="#dddddd", showbackground=True),
        yaxis=dict(title="x (mm)", backgroundcolor="white", gridcolor="#dddddd", showbackground=True),
        zaxis=dict(title="y (mm)", backgroundcolor="white", gridcolor="#dddddd", showbackground=True),
        bgcolor="white",
    ),
    legend=dict(itemsizing='constant', font=dict(size=11)),
    margin=dict(l=0, r=0, t=55, b=0),
    width=860, height=620,
    paper_bgcolor="white",
)
fig3d.show()

__Physics characterisation — what drives model performance?__

Augment per-event records with physics quantities, then investigate where the model struggles.

__Per-event accuracy vs physics quantities__

Each point is one val event. The black line is a **binned median**: events are sorted by the x-variable, divided into equal-count bins, and the median accuracy in each bin is plotted — a non-parametric way to show the trend without assuming a functional form.

In [ ]:
def binned_median(x, y, n_bins=15):
    """Sort events by x, split into equal-count bins, return (bin_centre, median_y)."""
    valid  = np.isfinite(x) & np.isfinite(y)
    x, y   = x[valid], y[valid]
    order  = np.argsort(x)
    xs, ys = x[order], y[order]
    chunks = np.array_split(np.arange(len(xs)), n_bins)
    return (np.array([xs[c].mean()      for c in chunks if len(c)]),
            np.array([np.median(ys[c])  for c in chunks if len(c)]))

panels = [
    (Enu_arr,  acc_arr, "$E_\\nu$ (GeV)",    "log"),
    (inel_arr, acc_arr, "Inelasticity $y$",  "linear"),
    (nn_arr,   acc_arr, "Nodes per event",   "linear"),
]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (xarr, yarr, xlabel, xscale) in zip(axes, panels):
    ax.scatter(xarr, yarr, c=C_BG, s=8, alpha=0.30, linewidths=0, rasterized=True)
    cx, cy = binned_median(xarr, yarr)
    ax.plot(cx, cy, color='k', lw=1.5, zorder=5)
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel("Per-event accuracy", fontsize=9)
    ax.set_xscale(xscale)
    ax.set_ylim(0, 1.05)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(labelsize=8)

plt.tight_layout()
plt.savefig(figures_path / "acc_vs_physics.png", dpi=350, bbox_inches='tight')
plt.show()

__Spatial distribution of misclassified nodes (pooled across all val events)__

2-D histograms of misclassified node positions. Left: background nodes called primary_EM_e. Right: primary_EM_e nodes called background. Reveals *where* in the detector the decision boundary breaks down.

In [ ]:
# Pool positions of misclassified nodes across all val events
pos_bg2em = []   # background → predicted primary_EM_e
pos_em2bg = []   # primary_EM_e → predicted background

for rec in records:
    pos, pred, true = rec['pos'], rec['pred'], rec['true']
    mask_bg2em = (true == 0) & (pred == 1)
    mask_em2bg = (true == 1) & (pred == 0)
    if mask_bg2em.any(): pos_bg2em.append(pos[mask_bg2em])
    if mask_em2bg.any(): pos_em2bg.append(pos[mask_em2bg])

pos_bg2em = np.concatenate(pos_bg2em, axis=0)
pos_em2bg = np.concatenate(pos_em2bg, axis=0)
print(f"bg→EM errors: {len(pos_bg2em):,} nodes")
print(f"EM→bg errors: {len(pos_em2bg):,} nodes")

# 2-D histograms: xz view (z on x-axis, x on y-axis) — matches event display convention
BINS = 60
error_sets = [
    (pos_bg2em, "bg → predicted primary_EM_e", C_EM, "Reds"),
    (pos_em2bg, "primary_EM_e → predicted bg", C_BG, "Blues"),
]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (pos_err, title, color, cmap) in zip(axes, error_sets):
    h, xe, ye, img = ax.hist2d(
        pos_err[:, 2], pos_err[:, 0],
        bins=BINS, cmap=cmap, density=True,
    )
    plt.colorbar(img, ax=ax, label="Density")
    ax.set_xlabel("$z$ (mm)", fontsize=9)
    ax.set_ylabel("$x$ (mm)", fontsize=9)
    ax.set_title(title, fontsize=9)
    ax.tick_params(labelsize=8)

plt.suptitle("Spatial distribution of misclassified nodes ($xz$ projection, pooled val set)",
             y=1.02, fontsize=10)
plt.tight_layout()
plt.savefig(figures_path / "error_spatial_density.png", dpi=350, bbox_inches='tight')
plt.show()

__PDG label breakdown of misclassified nodes__

For background nodes wrongly called primary_EM_e: which original particle types are being confused? Tells us whether the model struggles with secondary electrons (which genuinely look EM-like) vs hadronic hits vs muon hits.

In [ ]:
PDG_NAMES = {0: "other/hadronic", 1: "secondary_e", 2: "primary_EM_e", 3: "muon"}

# Pool all val nodes
all_pred = np.concatenate([r['pred']      for r in records])
all_true = np.concatenate([r['true']      for r in records])
all_pdg  = np.concatenate([r['pdg_label'] for r in records])

pdg_classes = sorted(np.unique(all_pdg))
bg_classes  = [p for p in pdg_classes if p != 2]   # background types only

# For each background PDG class: total nodes vs misclassified as primary_EM_e
fig, ax = plt.subplots(figsize=(7, 4))

for i, p in enumerate(bg_classes):
    mask     = all_pdg == p
    total    = int(mask.sum())
    confused = int(((all_pdg == p) & (all_pred == 1)).sum())
    correct  = total - confused
    rate     = confused / total if total > 0 else 0

    ax.bar(i, correct,  color='#cccccc', edgecolor='white', alpha=0.9,
           label='correctly classified' if i == 0 else None)
    ax.bar(i, confused, bottom=correct, color=C_EM, edgecolor='white', alpha=0.85,
           label='misclassified as primary_EM_e' if i == 0 else None)
    ax.text(i, total * 1.03, f"rate = {rate:.3f}\n(n = {total:,})",
            ha='center', va='bottom', fontsize=8)

ax.set_xticks(range(len(bg_classes)))
ax.set_xticklabels([PDG_NAMES.get(p, str(p)) for p in bg_classes], fontsize=9)
ax.set_ylabel("Node count", fontsize=9)
ax.legend(fontsize=8, frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(labelsize=8)

plt.tight_layout()
plt.savefig(figures_path / "bg_pdg_breakdown.png", dpi=350, bbox_inches='tight')
plt.show()

for p in bg_classes:
    mask = all_pdg == p
    rate = ((all_pdg == p) & (all_pred == 1)).sum() / mask.sum()
    print(f"  {PDG_NAMES.get(p, p):<20}: {int(mask.sum()):>10,} nodes,  confusion rate = {rate:.4f}")

__Secondary electron analysis__

For each true PDG class: how often does the model predict primary_EM_e? For secondary electrons (PDG=1) this is the key question — they are EM-like background nodes and the hardest to separate. Right panel shows the purity of primary_EM_e predictions (what fraction are actually primary_EM_e vs contamination).

In [ ]:
PDG_NAMES = {0: "other/hadronic", 1: "secondary_e", 2: "primary_EM_e", 3: "muon"}

# Pool all val nodes
all_pred = np.concatenate([r['pred']      for r in records])
all_true = np.concatenate([r['true']      for r in records])
all_pdg  = np.concatenate([r['pdg_label'] for r in records])

pdg_classes = sorted(np.unique(all_pdg))

# Left panel: per-PDG-class rate of being predicted as primary_EM_e
#   pdg==2 → this is recall;  pdg!=2 → this is the false-positive rate for that class
pred_pEM_rate = {}
class_counts  = {}
for p in pdg_classes:
    m = all_pdg == p
    pred_pEM_rate[p] = (all_pred[m] == 1).mean()
    class_counts[p]  = int(m.sum())

# Right panel: composition of nodes predicted as primary_EM_e (purity)
pred_pEM_mask    = all_pred == 1
pdg_in_pred_pEM  = all_pdg[pred_pEM_mask]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# ── Left: prediction rate per true PDG class ─────────────────────────────────
names  = [PDG_NAMES.get(p, str(p)) for p in pdg_classes]
rates  = [pred_pEM_rate[p]         for p in pdg_classes]
colors = [C_EM if p == 2 else ("#E17883" if p == 1 else "#aaaaaa") for p in pdg_classes]
bars   = axes[0].bar(names, rates, color=colors, alpha=0.85, edgecolor='white')
for bar, rate, p in zip(bars, rates, pdg_classes):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                 f"{rate:.3f}\n(n={class_counts[p]:,})",
                 ha='center', va='bottom', fontsize=7.5)
axes[0].set_ylabel("P(predicted primary_EM_e | true PDG class)", fontsize=9)
axes[0].set_title("Prediction rate per true particle type", fontsize=9)
axes[0].set_ylim(0, 1.25)
axes[0].axhline(0.5, color='gray', ls='--', lw=0.8, alpha=0.5)
axes[0].tick_params(axis='x', labelsize=8, rotation=15)
axes[0].spines[["top", "right"]].set_visible(False)

# ── Right: purity of primary_EM_e predictions ────────────────────────────────
comp_classes = sorted(np.unique(pdg_in_pred_pEM))
comp_names   = [PDG_NAMES.get(p, str(p)) for p in comp_classes]
comp_counts  = np.array([(pdg_in_pred_pEM == p).sum() for p in comp_classes], dtype=float)
comp_fracs   = comp_counts / comp_counts.sum()
comp_colors  = [C_EM if p == 2 else ("#E17883" if p == 1 else "#aaaaaa") for p in comp_classes]
bars2 = axes[1].bar(comp_names, comp_fracs, color=comp_colors, alpha=0.85, edgecolor='white')
for bar, frac, cnt in zip(bars2, comp_fracs, comp_counts):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                 f"{frac:.3f}\n({int(cnt):,})",
                 ha='center', va='bottom', fontsize=7.5)
axes[1].set_ylabel("Fraction of predicted primary_EM_e nodes", fontsize=9)
axes[1].set_title(f"Purity of primary_EM_e predictions  (n={pred_pEM_mask.sum():,})", fontsize=9)
axes[1].set_ylim(0, 1.25)
axes[1].tick_params(axis='x', labelsize=8, rotation=15)
axes[1].spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(figures_path / "secondary_e_analysis.png", dpi=350, bbox_inches='tight')
plt.show()

# Summary printout
if 1 in pred_pEM_rate:
    sec_n     = class_counts[1]
    sec_rate  = pred_pEM_rate[1]
    sec_contam = (pdg_in_pred_pEM == 1).sum() / pred_pEM_mask.sum()
    print(f"secondary_e:  {sec_n:,} nodes,  {sec_rate:.1%} predicted as pEM  "
          f"(contamination in pEM predictions: {sec_contam:.1%})")
if 2 in pred_pEM_rate:
    print(f"primary_EM_e: recall = {pred_pEM_rate[2]:.1%}")

__Inelasticity $y \\approx 0.5$ \u2014 worst-performing events (Task A)__

Select events with $y \\in [0.45,\\,0.55]$ and rank by per-event primary\_EM\_e F1.
For each event: true labels vs predicted labels ($xz$ projection), misclassified nodes
marked with diamonds, and a boundary/distributed confusion classification.

In [ ]:
# ── Filter intermediate-inelasticity events and compute per-event pEM F1 ────
Y_LO, Y_HI = 0.45, 0.55
N_SEL       = 4      # worst events to display

mid_records = []
for rec in records:
    y = rec['inelasticity']
    if np.isnan(y) or not (Y_LO <= y <= Y_HI):
        continue
    pred, true = rec['pred'], rec['true']
    tp  = int(((pred == 1) & (true == 1)).sum())
    fp  = int(((pred == 1) & (true == 0)).sum())
    fn  = int(((pred == 0) & (true == 1)).sum())
    n_em = int((true == 1).sum())
    if n_em == 0:
        continue
    denom  = 2 * tp + fp + fn
    f1_pEM = (2 * tp / denom) if denom > 0 else 0.0
    mid_records.append(dict(rec, f1_pEM=f1_pEM, tp=tp, fp=fp, fn=fn, n_em=n_em))

mid_records.sort(key=lambda r: r['f1_pEM'])
sel_mid = mid_records[:N_SEL]

print(f"Events with y \u2208 [{Y_LO}, {Y_HI}]: {len(mid_records)}  \u2192  showing worst {N_SEL}")
print()
print(f"{'idx':>5}  {'y':>6}  {'acc':>6}  {'f1_pEM':>8}  {'pEM_frac':>10}  "
      f"{'n_em':>6}  {'n_nodes':>8}  {'FP':>5}  {'FN':>5}")
for r in sel_mid:
    print(f"{r['idx']:>5}  {r['inelasticity']:>6.3f}  {r['acc']:>6.3f}  "
          f"{r['f1_pEM']:>8.3f}  {r['pEM_frac']:>10.3f}  "
          f"{r['n_em']:>6}  {r['n_nodes']:>8}  {r['fp']:>5}  {r['fn']:>5}")


In [ ]:
import matplotlib.patches as mpatches

C_BG_COL  = "#4477AA"   # blue  \u2014 background
C_EM_TRUE = "#CC3311"   # red   \u2014 primary_EM_e (true-label panel)
C_EM_PRED = "#44AA77"   # green \u2014 primary_EM_e (predicted-label panel)
C_FP_COL  = "#FFAA00"   # amber \u2014 false-positive highlight

fig, axes = plt.subplots(len(sel_mid), 2,
                          figsize=(11, 3.6 * len(sel_mid)),
                          gridspec_kw={"wspace": 0.35, "hspace": 0.55})
if len(sel_mid) == 1:
    axes = axes[np.newaxis, :]

for row, rec in enumerate(sel_mid):
    pos     = rec['pos']
    pred    = rec['pred']
    true    = rec['true']
    correct = pred == true
    y_val   = rec['inelasticity']
    f1      = rec['f1_pEM']
    pEM_frac= rec['pEM_frac']

    m_fn = (true == 1) & ~correct          # missed EM (FN)
    m_fp = (true == 0) & ~correct          # false positive (FP)
    m_em_ok = (true == 1) & correct

    # ── (a) True-label panel ────────────────────────────────────────────────
    ax = axes[row, 0]
    ax.scatter(pos[true == 0, 2], pos[true == 0, 0],
               c=C_BG_COL, s=4, alpha=0.20, linewidths=0, zorder=2, rasterized=True)
    if m_em_ok.any():
        ax.scatter(pos[m_em_ok, 2], pos[m_em_ok, 0],
                   c=C_EM_TRUE, s=7, alpha=0.55, linewidths=0, zorder=3, rasterized=True)
    if m_fn.any():
        ax.scatter(pos[m_fn, 2], pos[m_fn, 0],
                   c=C_EM_TRUE, s=32, marker='D', linewidths=0.9,
                   edgecolors='k', alpha=0.95, zorder=5, rasterized=True)
    if m_fp.any():
        ax.scatter(pos[m_fp, 2], pos[m_fp, 0],
                   c=C_FP_COL,  s=26, marker='D', linewidths=0.8,
                   edgecolors='k', alpha=0.90, zorder=5, rasterized=True)
    ax.set_xlabel("$z$ (mm)", fontsize=8)
    ax.set_ylabel("$x$ (mm)", fontsize=8)
    ax.set_title(
        f"(a) True  |  $y={y_val:.3f}$  "
        f"$F1_{{\\mathrm{{pEM}}}}={f1:.3f}$  pEM frac$={pEM_frac:.2f}$",
        fontsize=8, loc='left',
    )
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(labelsize=7)

    # ── (b) Predicted-label panel ───────────────────────────────────────────
    ax = axes[row, 1]
    m_pred_bg_ok = (pred == 0) & correct
    m_pred_em_ok = (pred == 1) & correct
    if m_pred_bg_ok.any():
        ax.scatter(pos[m_pred_bg_ok, 2], pos[m_pred_bg_ok, 0],
                   c=C_BG_COL, s=4, alpha=0.20, linewidths=0, zorder=2, rasterized=True)
    if m_pred_em_ok.any():
        ax.scatter(pos[m_pred_em_ok, 2], pos[m_pred_em_ok, 0],
                   c=C_EM_PRED, s=7, alpha=0.55, linewidths=0, zorder=3, rasterized=True)
    if m_fn.any():
        ax.scatter(pos[m_fn, 2], pos[m_fn, 0],
                   c=C_BG_COL, s=32, marker='D', linewidths=0.9,
                   edgecolors='k', alpha=0.95, zorder=5, rasterized=True)
    if m_fp.any():
        ax.scatter(pos[m_fp, 2], pos[m_fp, 0],
                   c=C_EM_PRED, s=26, marker='D', linewidths=0.8,
                   edgecolors='k', alpha=0.90, zorder=5, rasterized=True)

    # Boundary vs distributed: are FN nodes near the z-edges of the EM cluster?
    confusion_type = "N/A"
    if m_fn.any() and m_em_ok.any():
        z_em  = pos[true == 1, 2]
        z_rng = z_em.max() - z_em.min()
        if z_rng > 0:
            z_fn = pos[m_fn, 2]
            edge_frac = float(np.mean(
                (z_fn < z_em.min() + 0.25 * z_rng) |
                (z_fn > z_em.max() - 0.25 * z_rng)
            ))
            confusion_type = "boundary" if edge_frac > 0.55 else "distributed"

    ax.set_xlabel("$z$ (mm)", fontsize=8)
    ax.set_ylabel("$x$ (mm)", fontsize=8)
    ax.set_title(
        f"(b) Predicted  |  FP={rec['fp']}  FN={rec['fn']}  [{confusion_type}]",
        fontsize=8, loc='left',
    )
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(labelsize=7)

leg_handles = [
    mpatches.Patch(color=C_BG_COL,  alpha=0.6, label='background'),
    mpatches.Patch(color=C_EM_TRUE, alpha=0.6, label='primary_EM_e (true, red)'),
    mpatches.Patch(color=C_EM_PRED, alpha=0.6, label='primary_EM_e (predicted, green)'),
    mlines.Line2D([], [], marker='D', color='w', markerfacecolor='#888888',
                  markeredgecolor='k', markersize=7, linestyle='None',
                  label='misclassified (diamond)'),
    mpatches.Patch(color=C_FP_COL,  alpha=0.85, label='FP bg\u2192pEM (amber, panel a)'),
]
fig.legend(handles=leg_handles, loc='upper center', bbox_to_anchor=(0.5, 1.01),
           ncol=5, fontsize=7.5, frameon=False)
plt.suptitle(
    f"Worst-performing events with $y \\in [{Y_LO},\\,{Y_HI}]$ \u2014 "
    "ranked by primary\_EM\_e F1  ($xz$ projection)",
    y=1.05, fontsize=10,
)
plt.savefig(figures_path / "inelasticity_mid_events.png", dpi=350, bbox_inches='tight')
plt.show()

# ── Physical interpretation summary ──────────────────────────────────────────
print("\nPhysical summary (hypothesis: secondary EM shower confusable with primary at y~0.5):")
print(f"{'idx':>5}  {'y':>6}  {'pEM_frac':>10}  {'n_em':>6}  {'n_nodes':>8}  "
      f"{'F1':>6}  {'confusion':>12}")
for rec in sel_mid:
    pos_, pred_, true_ = rec['pos'], rec['pred'], rec['true']
    correct_ = pred_ == true_
    m_fn_    = (true_ == 1) & ~correct_
    m_em_ok_ = (true_ == 1) & correct_
    ct = "N/A"
    if m_fn_.any() and m_em_ok_.any():
        z_em_  = pos_[true_ == 1, 2]
        z_rng_ = z_em_.max() - z_em_.min()
        if z_rng_ > 0:
            z_fn_  = pos_[m_fn_, 2]
            ef     = float(np.mean(
                (z_fn_ < z_em_.min() + 0.25 * z_rng_) |
                (z_fn_ > z_em_.max() - 0.25 * z_rng_)
            ))
            ct = "boundary" if ef > 0.55 else "distributed"
    print(f"{rec['idx']:>5}  {rec['inelasticity']:>6.3f}  "
          f"{rec['pEM_frac']:>10.3f}  {rec['n_em']:>6}  {rec['n_nodes']:>8}  "
          f"{rec['f1_pEM']:>6.3f}  {ct:>12}")
